In [1]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.chains.llm import LLMChain

from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain.chains.router.multi_prompt_prompt import MULTI_PROMPT_ROUTER_TEMPLATE

In [2]:
# Template para o especialista em Química
quimica_template = ChatPromptTemplate.from_template("""Você é um químico muito experiente.
Você é excelente em responder perguntas sobre química de forma clara e objetiva.
Você tem um grande entendimento sobre reações químicas, elementos, compostos,
e a relação entre a estrutura molecular e as propriedades dos materiais.
Quando você não sabe a resposta para uma pergunta, você admite que não sabe.

Aqui está uma pergunta: {input}""")

# Template para o especialista em Geografia
geografia_template = ChatPromptTemplate.from_template("""Você é um geógrafo muito bem informado.
Você tem um vasto conhecimento sobre os processos naturais, geografia humana,
clima, relevo e interações entre o ambiente e a sociedade.
Você é habilidoso em explicar como fatores físicos e humanos afetam
o mundo ao nosso redor.

Aqui está uma pergunta: {input}""")

# Template para o especialista em Biologia
biologia_template = ChatPromptTemplate.from_template("""Você é um biólogo muito capacitado.
Você tem um grande conhecimento sobre os seres vivos, suas estruturas, funções
e a interação entre diferentes organismos e seus ambientes.
Você é excelente em explicar conceitos de biologia de maneira clara,
tanto para iniciantes quanto para estudantes avançados.

Aqui está uma pergunta: {input}""")


In [3]:
prompts_infos = [
    {
        "name": "Química",
        "description": "Ideal para responder perguntas sobre química",
        "prompt_template": quimica_template
    },
    {
        "name": "Geografia",
        "description": "Ideal para responder perguntas sobre geografia",
        "prompt_template": geografia_template
    },
    {
        "name": "Biologia",
        "description": "Ideal para responder perguntas sobre biologia",
        "prompt_template": biologia_template
    }
]



In [4]:
chat = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [5]:
chains_destinos = {}
for info in prompts_infos:
    chain = LLMChain(llm=chat, prompt=info["prompt_template"], verbose=True)
    chains_destinos[info["name"]] = chain
    
chains_destinos

{'Química': LLMChain(verbose=True, prompt=ChatPromptTemplate(input_variables=['input'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='Você é um químico muito experiente.\nVocê é excelente em responder perguntas sobre química de forma clara e objetiva.\nVocê tem um grande entendimento sobre reações químicas, elementos, compostos,\ne a relação entre a estrutura molecular e as propriedades dos materiais.\nQuando você não sabe a resposta para uma pergunta, você admite que não sabe.\n\nAqui está uma pergunta: {input}'))]), llm=ChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x109ea3a10>, async_client=<openai.resources.chat.completions.AsyncCompletions object at 0x109ed7620>, model_name='gpt-3.5-turbo-0125', openai_api_key=SecretStr('**********'), openai_proxy='')),
 'Geografia': LLMChain(verbose=True, prompt=ChatPromptTemplate(input_variables=['input'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(inp

In [6]:
destinos = [f"{p['name']}: {p['description']}" for p in prompts_infos]
destinos_str = "\n".join(destinos)
print(destinos_str)

Química: Ideal para responder perguntas sobre química
Geografia: Ideal para responder perguntas sobre geografia
Biologia: Ideal para responder perguntas sobre biologia


In [8]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinos_str
)
print(router_template)

Given a raw text input to a language model select the model prompt best suited for the input. You will be given the names of the available prompts and a description of what the prompt is best suited for. You may also revise the original input if you think that revising it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}
```

REMEMBER: "destination" MUST be one of the candidate prompt names specified below OR it can be "DEFAULT" if the input is not well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
Química: Ideal para responder perguntas sobre química
Geografia: Ideal para responder perguntas sobre

In [9]:
router_template = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser()
)
router_chain = LLMRouterChain.from_llm(chat, router_template, verbose=True)

In [10]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=chat, prompt=default_prompt, verbose=True)
chain = MultiPromptChain(
    router_chain=router_chain,
    destination_chains=chains_destinos,
    default_chain=default_chain,
    verbose=True
)

In [13]:
chain.invoke({"input": "O que é o El Niño?"})



> Entering new MultiPromptChain chain...


> Entering new LLMRouterChain chain...

> Finished chain.
Geografia: {'input': 'O que é o El Niño?'}

> Entering new LLMChain chain...
Prompt after formatting:
Human: Você é um geógrafo muito bem informado.
Você tem um vasto conhecimento sobre os processos naturais, geografia humana,
clima, relevo e interações entre o ambiente e a sociedade.
Você é habilidoso em explicar como fatores físicos e humanos afetam
o mundo ao nosso redor.

Aqui está uma pergunta: O que é o El Niño?

> Finished chain.

> Finished chain.


{'input': 'O que é o El Niño?',
 'text': 'O El Niño é um fenômeno climático que ocorre de forma periódica no Oceano Pacífico, caracterizado pelo aquecimento anormal das águas superficiais no Oceano Pacífico Equatorial. Esse aquecimento altera os padrões de vento e de correntes oceânicas, o que tem impacto direto no clima de diversas regiões do mundo. O El Niño pode resultar em secas em algumas áreas e enchentes em outras, afetando a agricultura, a pesca, a disponibilidade de água potável e a ocorrência de desastres naturais. É um fenômeno complexo e de grande importância para a compreensão e previsão do clima global.'}

In [14]:
chain.invoke({"input": "Para que serve a tabela periódica?"})



> Entering new MultiPromptChain chain...


> Entering new LLMRouterChain chain...

> Finished chain.
Química: {'input': 'Para que serve a tabela periódica?'}

> Entering new LLMChain chain...
Prompt after formatting:
Human: Você é um químico muito experiente.
Você é excelente em responder perguntas sobre química de forma clara e objetiva.
Você tem um grande entendimento sobre reações químicas, elementos, compostos,
e a relação entre a estrutura molecular e as propriedades dos materiais.
Quando você não sabe a resposta para uma pergunta, você admite que não sabe.

Aqui está uma pergunta: Para que serve a tabela periódica?

> Finished chain.

> Finished chain.


{'input': 'Para que serve a tabela periódica?',
 'text': 'A tabela periódica é uma ferramenta fundamental na química, pois organiza os elementos químicos de acordo com suas propriedades físicas e químicas. Ela fornece informações importantes sobre a estrutura atômica dos elementos, como o número atômico, massa atômica, configuração eletrônica, entre outros.\n\nAlém disso, a tabela periódica permite prever o comportamento dos elementos em reações químicas, identificar semelhanças e diferenças entre eles, e facilitar a criação de novos compostos químicos. Em resumo, a tabela periódica é essencial para o estudo e compreensão da química, sendo uma ferramenta indispensável para químicos, estudantes e pesquisadores da área.'}

In [15]:
chain.invoke({"input": "Para que serve a fotossintese?"})



> Entering new MultiPromptChain chain...


> Entering new LLMRouterChain chain...

> Finished chain.
Biologia: {'input': 'Para que serve a fotossíntese?'}

> Entering new LLMChain chain...
Prompt after formatting:
Human: Você é um biólogo muito capacitado.
Você tem um grande conhecimento sobre os seres vivos, suas estruturas, funções
e a interação entre diferentes organismos e seus ambientes.
Você é excelente em explicar conceitos de biologia de maneira clara,
tanto para iniciantes quanto para estudantes avançados.

Aqui está uma pergunta: Para que serve a fotossíntese?

> Finished chain.

> Finished chain.


{'input': 'Para que serve a fotossíntese?',
 'text': 'A fotossíntese é um processo fundamental realizado por plantas, algas e algumas bactérias que converte a energia solar em energia química, na forma de glicose. Esse processo é essencial para a sobrevivência dos seres vivos, pois fornece a principal fonte de alimento e energia para a maioria dos organismos heterotróficos. Além disso, a fotossíntese também é responsável pela produção de oxigênio, que é fundamental para a respiração dos seres vivos aeróbicos. Em resumo, a fotossíntese é responsável por manter o equilíbrio da cadeia alimentar e por manter a quantidade de oxigênio na atmosfera, sendo um processo vital para a vida na Terra.'}